# EDA (Exploratory Data Analysis) of the Abalone dataset

The age of an abalone is traditionally obtained by cutting the shell, staining it and counting its rings under a
microscope. This notebook explores whether simple physical measurements carry enough signal to predict the number
of rings (`Rings`), and which preprocessing decisions the modelling step will need.

Each section ends with its own conclusions; the final section gathers them into decisions for the modelling notebook.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 500)
sns.set_theme(style="whitegrid")

# Data

Column definitions, from the [dataset description](https://archive.ics.uci.edu/dataset/1/abalone) (UCI, also
reproduced on Kaggle):

| Column | Description |
|---|---|
| `Sex` | M (male), F (female) or I (infant, too young for its sex to be determined) |
| `Length` | Longest shell measurement |
| `Diameter` | Shell measurement perpendicular to the length |
| `Height` | Height with the meat in the shell |
| `Whole weight` | Weight of the whole abalone |
| `Shucked weight` | Weight of the meat |
| `Viscera weight` | Gut weight, after bleeding |
| `Shell weight` | Shell weight, after being dried |
| `Rings` | Number of rings: the target (age in years = `Rings` + 1.5) |

Continuous values were scaled in the original dataset (divided by 200), so units are not meaningful but ratios are.

In [ ]:
DATA_PATH = Path("../data/abalone.csv")
TARGET_COL = "Rings"
CATEGORICAL_COLS = ["Sex"]


def load_data(path: Path) -> pd.DataFrame:
    """Load the Abalone CSV dataset."""
    return pd.read_csv(path)


df = load_data(DATA_PATH)
NUMERICAL_COLS = [col for col in df.columns if col not in CATEGORICAL_COLS + [TARGET_COL]]

print(f"Shape: {df.shape}")
df.head()

## 1 - First look at the data

Before any visualisation, check that the data is complete and consistent: column types, missing values, duplicates
and value ranges.

In [ ]:
df.info()

In [ ]:
pd.DataFrame(
    {
        "missing_values": df.isna().sum(),
        "n_unique": df.nunique(),
    }
)

Nothing is missing. `Sex` has 3 distinct values: the third category `I` stands for infant (see the table above).

In [ ]:
df["Sex"].value_counts()

The three categories are balanced, so `Sex` can be one-hot encoded without any rare-category handling.
Next, look for duplicates and for values that are out of the expected range.

In [ ]:
print(f"Duplicated rows: {df.duplicated().sum()}")
df.describe().T

`Height` has a minimum of 0 and a maximum of 1.13, far above its 75th percentile (0.165). Since all measurements
are physical sizes or weights, they must be strictly positive.

In [ ]:
(df[NUMERICAL_COLS] <= 0).sum()

In [ ]:
df[df["Height"] <= 0]

Two infants have a zero height while their other measurements are normal: these are recording errors.

The weights can also be cross-checked. Meat, viscera and shell are parts of the whole abalone, so none of them,
taken alone, can weigh more than the whole animal.

In [ ]:
WEIGHT_PART_COLS = ["Shucked weight", "Viscera weight", "Shell weight"]

for col in WEIGHT_PART_COLS:
    print(f"{col} > Whole weight: {(df[col] > df['Whole weight']).sum()} rows")

In [ ]:
df[(df[WEIGHT_PART_COLS].gt(df["Whole weight"], axis=0)).any(axis=1)]

5 abalones break this rule: 4 have more meat than their whole weight, and 1 has a shell 2.6 times heavier than the
whole animal (it is also one of the two zero-height abalones). These are recording errors.

A stricter rule was considered: the *sum* of the three parts should not exceed the whole weight. It was rejected
because it relies on information the dataset does not provide:

- the description does not say whether the viscera are included in the shucked weight; if they are, the viscera are
  counted twice in the sum, and a sum above the whole weight is expected;
- the weighing protocol (order, delay between weighings, water absorbed or lost) is unknown.

The single-part rule only relies on the definition of the columns. It is also the rule used in a
[DataCamp course](https://campus.datacamp.com/courses/r-for-sas-users/data-wrangling-2?ex=13) on this dataset.

The shell dimensions can be cross-checked the same way: the length is by definition the longest measurement, so it
should never be smaller than the diameter. Look also at the extreme heights spotted in the summary statistics.

In [ ]:
print(f"Diameter > Length: {(df['Diameter'] > df['Length']).sum()} rows")
print(f"Median height-to-length ratio: {(df['Height'] / df['Length']).median():.2f}")
df.assign(height_to_length=df["Height"] / df["Length"]).nlargest(5, "Height")[
    ["Sex", "Length", "Diameter", "Height", "height_to_length", "Whole weight", TARGET_COL]
]

A typical abalone is about 4 times longer than high (median height-to-length ratio of 0.26). The two tallest
abalones (1.13 and 0.515) are 2 to 4 times taller than the next ones (0.25) with an ordinary length and weight: they
are most likely typos, as is the abalone whose diameter is twice its length.

The rules found above are gathered in one function, which will be reused in the modelling notebook.

In [ ]:
MAX_HEIGHT = 0.5


def get_erroneous_rows(df: pd.DataFrame) -> pd.Series:
    """Flag physically impossible records (zero or extreme height, inconsistent dimensions or weights)."""
    return (
        (df["Height"] <= 0)
        | (df["Height"] > MAX_HEIGHT)
        | (df["Diameter"] > df["Length"])
        | df[WEIGHT_PART_COLS].gt(df["Whole weight"], axis=0).any(axis=1)
    )


erroneous_rows = get_erroneous_rows(df)
print(f"Erroneous rows: {erroneous_rows.sum()} ({erroneous_rows.mean():.1%} of the data)")

**Conclusions - first look**

- 4,177 abalones, 9 columns: 1 categorical feature (`Sex`, with balanced M / F / I categories), 7 continuous physical
  measurements and the integer target `Rings`.
- No missing values and no duplicated rows: no imputation is needed.
- 9 erroneous records (0.2% of the data) will be removed in the modelling preprocessing:
  - 2 abalones with `Height = 0`;
  - 2 abalones with an implausible height (1.13 and 0.515);
  - 1 abalone with a diameter larger than its length;
  - 5 abalones with a meat or shell weight larger than the whole weight (one of them also has a zero height).
- The rest of the analysis is done on the cleaned data.

In [ ]:
df = df[~erroneous_rows].reset_index(drop=True)
print(f"Shape after cleaning: {df.shape}")

## 2 - The target: `Rings`

The target choice drives the modelling: its distribution tells whether a regression is appropriate, which metric to
use and whether some ages will be hard to predict because they are rare.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.countplot(data=df, x=TARGET_COL, color="tab:blue", ax=ax)
ax.set_title("Distribution of the number of rings")
plt.show()

In [ ]:
print(f"Skewness: {df[TARGET_COL].skew():.2f}")
print(f"Share of abalones with 7 to 12 rings: {df[TARGET_COL].between(7, 12).mean():.0%}")
print(f"Share of abalones with more than 15 rings: {(df[TARGET_COL] > 15).mean():.0%}")
df[TARGET_COL].value_counts().sort_index().to_frame().T

The distribution is unimodal around 9-10 rings with a long right tail: old abalones are rare. Some ring counts have
fewer than 10 examples, which rules out treating each count as a separate class. Since the counts are ordered
integers, predicting them as a continuous value is the natural choice.

The age in years is `Rings + 1.5`, a constant offset: predicting `Rings` or the age is strictly equivalent, so the
model predicts `Rings` directly.

To compare future models, compute the error of a naive model that always predicts the same value.

In [ ]:
naive_rmse = np.sqrt(((df[TARGET_COL] - df[TARGET_COL].mean()) ** 2).mean())
naive_mae = (df[TARGET_COL] - df[TARGET_COL].median()).abs().mean()

print(f"RMSE when always predicting the mean ({df[TARGET_COL].mean():.1f}): {naive_rmse:.2f} rings")
print(f"MAE when always predicting the median ({df[TARGET_COL].median():.0f}): {naive_mae:.2f} rings")

**Conclusions - target**

- `Rings` ranges from 1 to 29, centred on 9-10, and is right-skewed: most abalones have between 7 and 12 rings,
  very old ones are rare.
- The task is framed as a **regression** on `Rings`: the values are ordered, and rare counts make classification
  impractical.
- Rare old abalones will probably be the hardest to predict; the RMSE, which penalises large errors, will reflect
  this, while the MAE is easier to interpret (average error in rings, i.e. in years).
- A naive constant model reaches an RMSE of about 3.2 rings: any useful model must clearly beat this baseline.

## 3 - Numerical features

The 7 measurements are the inputs of the model. Look first at their individual distributions (skewness, remaining
outliers), then at how each of them relates to `Rings`.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, NUMERICAL_COLS):
    sns.histplot(data=df, x=col, bins=40, ax=ax)
    ax.set_title(f"{col} (skew = {df[col].skew():.2f})")
    ax.set_xlabel("")
axes.flat[-1].set_visible(False)
fig.suptitle("Distribution of the numerical features")
plt.tight_layout()
plt.show()

The dimensions (`Length`, `Diameter`, `Height`) are slightly left-skewed, while the weights are right-skewed. This
is expected: a weight grows roughly like a volume, i.e. like a length cubed. The scales also differ (dimensions
below 0.8, `Whole weight` up to 2.8), which matters for regularised linear models.

Now look at how each feature relates to the target.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, NUMERICAL_COLS):
    sns.scatterplot(data=df, x=col, y=TARGET_COL, alpha=0.2, s=10, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
axes.flat[-1].set_visible(False)
fig.suptitle("Rings as a function of each numerical feature")
plt.tight_layout()
plt.show()

All features increase with `Rings`, but the relationships are curved for the weights, and the spread of `Rings`
widens as abalones get bigger: once an abalone stops growing, its size says little about how old it is. Pearson's
coefficient measures linear relationships only, so it is compared with Spearman's, which captures any monotonic one.

In [ ]:
pd.DataFrame(
    {
        "pearson": df[NUMERICAL_COLS].corrwith(df[TARGET_COL], method="pearson"),
        "spearman": df[NUMERICAL_COLS].corrwith(df[TARGET_COL], method="spearman"),
    }
).sort_values("spearman", ascending=False).round(2)

All features are moderately correlated with `Rings` (0.5 to 0.7). Spearman is always higher than Pearson, which
confirms the curved relationships seen above. `Shell weight` is the most informative feature, `Shucked weight` the
least, possibly because the shell keeps thickening with age while the amount of meat depends more on each
abalone's condition.

To check how much the prediction error will depend on the size, compare the spread of `Rings` across size groups.

In [ ]:
size_groups = pd.qcut(df["Shell weight"], q=5, labels=["very small", "small", "medium", "large", "very large"])
df.groupby(size_groups, observed=True)[TARGET_COL].agg(["count", "mean", "std", "min", "max"]).round(1)

The standard deviation of `Rings` almost doubles from the smallest to the largest abalones (1.8 to 3.3 rings), and
large abalones can have anything from 7 to 29 rings. Young abalones will be predicted much more accurately than old
ones.

**Conclusions - numerical features**

- Every measurement carries information about the age (correlations between 0.5 and 0.7), `Shell weight` being the
  strongest predictor.
- The relationships are monotonic but not linear, especially for the weights: a linear regression on the raw features
  is a reasonable first model, but transformations (e.g. log of the weights) or non-linear models should do better.
- The spread of `Rings` grows with the size of the abalone: the error will be concentrated on large, old abalones, and
  there is an irreducible part of it since size stops being informative once growth slows down.
- Features have different scales: they should be standardised for regularised linear models.

## 4 - The categorical feature: `Sex`

`Sex` is the only non-physical measurement. Infants are by definition young, so this feature should be linked to
the age; the question is whether it brings information that the size does not already provide.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df, x="Sex", y=TARGET_COL, order=["I", "M", "F"], ax=ax)
ax.set_title("Rings by sex")
plt.show()

df.groupby("Sex")[TARGET_COL].describe().round(1)

Infants have about 3 rings fewer than adults (median of 8 against 10), while males and females have very similar
distributions. But infants are also smaller, so this gap could simply reflect their size. To separate the two
effects, compare the sexes within the size groups defined in section 3.

In [ ]:
pd.concat(
    {
        "mean_rings": df.pivot_table(index=size_groups, columns="Sex", values=TARGET_COL, aggfunc="mean", observed=True),
        "count": df.pivot_table(index=size_groups, columns="Sex", values=TARGET_COL, aggfunc="count", observed=True),
    },
    axis=1,
).round(1)

At the same size, infants still have fewer rings than adults among small abalones (6.4 against 7.8 to 8.3 for the
smallest group): they are young abalones that grew fast, rather than old abalones that stayed small. The gap
shrinks for medium abalones and disappears for large ones, and the few large infants (18 in the largest group) are not numerous enough to
conclude.

**Conclusions - `Sex`**

- Infants are clearly younger than adults; males and females are almost indistinguishable.
- `Sex` brings information beyond the size for small abalones: it should be kept as a feature and one-hot encoded.
- Since M and F behave alike, an `is_infant` flag would carry most of the signal; keeping the 3 categories is simpler
  and costs only one extra column.

## 5 - Correlations between features

Section 3 showed that each measurement is related to the age. Since they all describe the size of the same animal,
they are probably strongly related to each other too, which affects how a linear model can use them.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[NUMERICAL_COLS + [TARGET_COL]].corr(), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title("Pearson correlations between numerical features and the target")
plt.show()

All features are correlated with each other between 0.84 and 0.99 (`Length` and `Diameter` are almost
interchangeable), much more than any of them with `Rings` (0.42 to 0.63). In other words, the 7 measurements mostly
describe one thing, the size of the abalone, and bring only small complementary signals (for example the proportion
of meat versus shell).

**Conclusions - correlations between features**

- The features are highly collinear. For a linear regression this does not harm the predictions, but makes the
  individual coefficients unstable and hard to interpret (some may even become negative).
- All features are kept for the first model: the goal is prediction, not interpretation. Ridge regularisation is a
  simple way to stabilise the coefficients and can be compared with the plain linear regression.
- Ratios between weights (e.g. shell weight / whole weight) could capture the complementary signals; this is left as
  a possible improvement.

## 6 - Conclusions and decisions for the modelling

**Data quality**

- The dataset is complete (no missing values, no duplicates). 9 physically impossible records (0.2%) are removed
  with `get_erroneous_rows`: zero or extreme height, diameter larger than length, meat or shell heavier than the
  whole abalone.

**Target**

- The task is a regression on `Rings` (1 to 29, centred on 9-10, right-skewed). Old abalones are rare and will be the
  hardest to predict.
- Metrics: RMSE (main metric, penalises large errors) and MAE (average error in rings, i.e. in years). A model
  always predicting the mean reaches an RMSE of 3.22 rings: this is the baseline to beat.

**Features**

- All 7 measurements are informative (`Shell weight` the most), monotonically but non-linearly related to `Rings`,
  and highly correlated with each other.
- `Sex` is kept and one-hot encoded: infants are younger than adults of the same size.
- Numerical features are standardised, so that regularised models treat them equally.

**Modelling plan**

1. Preprocessing: remove erroneous rows, one-hot encode `Sex`, standardise numerical features, all packaged with the
   model in a scikit-learn `Pipeline`.
2. Models compared in MLflow: constant baseline, linear regression (required first version), Ridge, and a non-linear
   model (e.g. random forest) to measure how much the non-linear relationships matter.
3. Evaluation on a held-out test set, with the error expected to grow for old abalones.